# Overview and main features

**1. Import packages**

**2. Load data and feature engineering:**
* Read csv files.
* Basic data cleanup (Replace/remove/extend non-plausible or missing values)
* Feature engineering:
    * Physics-based feature engineering
    * Declare low cardinality integer features as categorical
    * Create N-grams of the categorical features (*N_GRAMS == [list of selected N-grams]*) and add (*N_GRAMS_EXT == True*) them to the original ones or threat them seperately for later preproccesing.
    * Define categorical/numerical features
* Configure data spliting strategy for k-fold (*FOLD == k*) training. Spliting can be done with a stratified/grouped (*STRAT == True* / *GROUP == True*) or simple K-Fold splitter. As reference for the optional stratification either a calculated multilabel (based on selected features and/or labels) or simply the label can be used (*EXTENDED_STRAT == False*).

**3. Explore data:**
* List of dataset columns including data types and number of non-zero elements.
* Show number of unique elements of categorical features.
* Show probability distribution of numerical features.
* Numerical feature statistics
* Correlation analysis of numerical features
* Compare probabilty distribution of features and labels between train, validation and test sets (based on a single fold).

**4. Preprocess data:**
* The dataset includes 2 main types of data: numerical features and categorical features.
* Helping functions for calculating and adding statistical features (mean/std/skew/median/min/max/count) of target values on categorical and bucketized numerical features.
* Define preprocessing pipelines. Selected preprocessing pipelines will be merged with a ColumnTransformer, possible pipelines are:
    * MinMax/standard/robust scaling of numerical features
    * Logaritmic/square/cube/square root/cube root transformations of numerical features
    * KBins discretization of numerical features
    * Ordinal/onehot or no encoding of categorical features
    * Principal component analysis (PCA) of numerical features
    * Linear Discriminant Analysis (LDA) of numerical features
    * Statistical features (mean/std/skew/median/min/max/count) for categorical and bucketized numerical features
* Preprocessed features can be passed into a feature selection model to extract most relevant features (*MAX_FEAT != None*). The maximum number of features can be configured (*MAX_FEAT*).
* Data preprocessing function allowing a leakage-free data preprocessing of folds.
* Casting function to convert categorical features into 'category' datatype for choosen estimators being able to handle it.
* External dataset can be merged with the compeition dataset for each fold seperately (*ADD_EXTERN_DATA == True*).

**5. Define model space for ML methods:**
* Definition of model space with a large number of possible ML methods:
    * Linear models: SGD
    * Support vector machine model: SVC
    * Ensemble models: RandomForest / GradientBoosting / AdaBoost / HistGradientBoosting / XGB / LGBM / CatBoost
    * Other models: KNeighbors
    * Neural network based models: RealMLP
* Definition of model parameter sets for both training and tuning interfaces.
* GPU acceleration can be activated (if supported by choosen ML method(s)) by *GPU_ACC*.

**6. Training:**
* Simple fitting or tuning (*TUNING == True*) of choosen models (*EST_IDS*) on whole trainval dataset with K-Fold method. Optionally, different seeds for the folds can be used (*MULTI_SEED == True*).
* Early stopping and the usage of category data type can be configured (*EST_IDS_W_EARLYSTOPPING/EST_IDS_W_CAT_FEAT*).
* Show single balanced accuracy scores for each folds and average balanced accuracy scores for each estimator over the folds during training.
* Stack (*STACKING == True*) or find individual weights for each estimator (*WEIGHTING == True*) .

**7. Evaluation:**
* Calculate balanced accuracy scores for each estimators and mean balanced accuracy score over all estimators based on accumulated Out-of-Fold (oof) samples.
* Show balanced accuracy scores of subcategory subsets.
* Show worse balanced accuracy scores of multicategory subsets.
* Show feature importances.

**8. Submission:**
* Calculate mean predictions either on base of probabilities (PROB_MEAN == True) or classes and create submission.csv file.
* If more than 1 model have been choosen in EST_IDS (Ensemble solution) then the prediction will be an average of the predictions of the single models.

Note: A similar notebook have been linked as baseline for regression tasks.

# 1. Import packages

In [ ]:
## Import packages

# General purpose modules
import time
import warnings
from copy import deepcopy
from importlib.metadata import version
from itertools import combinations

# Data handling and visualization modules
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image

# Skikit-learn preprocessing and evaluation modules
import sklearn
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, StratifiedGroupKFold
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, KBinsDiscretizer, TargetEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

# Skikit-learn ML modules
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import SGDClassifier, LogisticRegression

# Further ML modules
import xgboost as xgboost
import lightgbm as lightgbm
import catboost as catboost
import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier

# Show version numbers
print(f'Scikit-learn version: {sklearn.__version__}')
print(f'XGBoost version: {xgboost.__version__}')
print(f'LightGBM version: {lightgbm.__version__}')
print(f'CatBoost version: {catboost.__version__}')
print(f'PyTorch  version: {torch.__version__}')
print(f'PyTabKit  version: {version('pytabkit')}')

# 2. Load data and feature engineering

In [ ]:
## Read csv files

trainval = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
extern_data = pd.read_csv('/kaggle/input/datasets/ziya07/college-student-health-behavior-dataset/student_health_dataset_50k.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

In [ ]:
## Basic data cleanup: Replace/remove/extend non-plausible or missing values

#trainval = trainval.dropna().reset_index(drop=True)
#extern_data = extern_data.dropna().reset_index(drop=True)

In [ ]:
## Handcrafted feature engineering

# Physics-based feature engineering
orig_num_columns = [col for col in make_column_selector(dtype_include=['float', 'int'])(trainval) if col not in ['id']]
def feature_engineering(df):
    df = df.copy()
    eps = 1e-6
    # # 1.) Sleep-Related Features
    # df['sleep_deficit'] = np.maximum(0, 7 - df.sleep_duration)
    # df['sleep_excess'] = np.maximum(0, df.sleep_duration - 9)
    # df['sleep_score'] = df.sleep_duration * df.sleep_quality.map({
    #     'poor':0.6, 'average':0.8, 'good':1.0})
    
    # # 2.) Activity Features
    # df["calories_per_step"] = df.calorie_expenditure / (df.step_count + 1)
    # df["exercise_intensity"] = df.calorie_expenditure / (df.exercise_duration + 1)
    # df["steps_per_min"] = df.step_count / (df.exercise_duration + 1)
    return df

trainval = feature_engineering(trainval)
extern_data = feature_engineering(extern_data)
test = feature_engineering(test)

In [ ]:
## General feature engineering

# Target column
target = 'health_condition'
label_mapping = {'unhealthy': 0, 'at-risk': 1, 'fit': 2}
reverse_label_mapping = {v: k for k, v in label_mapping.items()}
trainval[target] = trainval[target].map(label_mapping).astype(np.uint8)
extern_data[target] = extern_data[target].map(label_mapping).astype(np.uint8)

# Declare low cardinality integer features as categorical
columns_to_cat = []
for df in [trainval, extern_data, test]:
    df[columns_to_cat] = df[columns_to_cat].astype('object')
orig_cat_columns = [col for col in make_column_selector(dtype_include='object')(trainval) if col not in ['class']]

# Create n-grams from selected categorical columns  
N_GRAMS = []         # Choose n-grams to derive
N_GRAMS_EXT = True    # Choose whether to add n-grams to categorical features
cat_columns2ngrams = ['spectral_type', 'galaxy_population'] # Selected categorical features for n-grams generation

def create_ngrams(df, n_grams=N_GRAMS):
    ngram_columns_dict = {} # Collect and concat the new columns to avoid memory fragmentation issues
    for n in n_grams:
        for ngram in combinations(cat_columns2ngrams, n):
            ngram_columns_dict[f'{n}gram_'+'_'.join(ngram)] = df[list(ngram)].astype(str).agg('_'.join, axis=1)
    ngram_columns_df = pd.DataFrame(ngram_columns_dict)
    updated_df = pd.concat([df, ngram_columns_df], axis=1)
    return updated_df
    
trainval = create_ngrams(trainval)
extern_data = create_ngrams(extern_data)
test = create_ngrams(test)

# Define categorical features
ngram_cat_columns = trainval.filter(like='gram_').columns.tolist()
cat_columns = (orig_cat_columns + ngram_cat_columns) if N_GRAMS_EXT else orig_cat_columns
print(f'Number of original categorical features: {len(orig_cat_columns)}')
print(f'Number of extended categorical features: {len(cat_columns)}')

# Define numerical features
num_columns = [col for col in make_column_selector(dtype_include=['float', 'int'])(trainval) if col not in ['id', target]]
print(f'Number of original numerical features: {len(orig_num_columns)}')
print(f'Number of extended numerical features: {len(num_columns)}')

In [ ]:
## Configure data spliting strategy for k-fold training

STRAT = True              # Use stratification for data spliting
EXTENDED_STRAT = False    # Stratification is based on multiple features
GROUP = False             # Use grouping for data spliting
FOLDS = 5                 # Number of folds for k-fold training

# Create multilabel column for evaluation purposes
multicat_encoder = LabelEncoder()
multicat_eval = ['stress_level', 'physical_activity_level']
trainval['multicat'] = multicat_encoder.fit_transform(trainval[multicat_eval].astype(str).agg('_'.join, axis=1)).astype('uint16')

# Determine stratification bins
strat_bin_encoder = LabelEncoder()
strat_cols = ['stress_level', 'physical_activity_level'] + [target]
trainval['strat_bin'] = strat_bin_encoder.fit_transform(trainval[strat_cols].astype(str).agg('_'.join, axis=1)).astype('uint16')

# Merge rare multicats into one category
rare_strat_bins = trainval['strat_bin'].value_counts()[trainval['strat_bin'].value_counts() < FOLDS].index.tolist()
if len(rare_strat_bins)>0:
    trainval['strat_bin'] = trainval['strat_bin'].replace(rare_strat_bins, rare_strat_bins[0])

# Experimental data spliting to check spliting quality
skf = ((StratifiedGroupKFold if GROUP else StratifiedKFold) if STRAT else (GroupKFold if GROUP else KFold))(
    n_splits=FOLDS, shuffle=True, random_state=42)
strat_col = 'strat_bin' if EXTENDED_STRAT else target
group_col = 'id'
train_idx, val_idx = next(skf.split(trainval, trainval[strat_col], trainval[group_col]))
train = trainval.iloc[train_idx].reset_index(drop=True)
val = trainval.iloc[val_idx].reset_index(drop=True)

# Verify dataset sizes of first fold
print(f'Total rows:   {len(trainval)}')
print(f'Dev train:    {len(train)} ({len(train)/len(trainval):.2%})')
print(f'Dev valid:    {len(val)} ({len(val)/len(trainval):.2%})')
print('-'*80, end='\n\n')

# Size of stratification bins
print(trainval['strat_bin'].value_counts().tail())
print(f'Number of unique elements in strat_bin column: {len(trainval['strat_bin'].unique())}')
print('-'*80, end='\n\n')
print(trainval['multicat'].value_counts().tail())
print(f'Number of unique elements in multicat column: {len(trainval['multicat'].unique())}')

# 3. Explore data

In [ ]:
## Explore train dataset

print('List of dataset columns including data types and number of non-zero elements: ', end='\n\n')
train.info()
print('-'*80, end='\n\n')
print('List of dataset columns including NaN values:')
nan_cols = [col for col in train.columns if pd.isna(train[col]).any()]
if nan_cols:
    for col in nan_cols: print(col)
else:
    print('No missing values found')

In [ ]:
## Explore categorical features

n_diags = len(orig_cat_columns)
n_cols = min(4, n_diags)
n_rows = (n_diags + n_cols - 1) // n_cols
fig1, axes1 = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
axes1 = axes1.flatten()
detailed_cols = []

for idx, cat in enumerate(orig_cat_columns):
    value_counts = train[cat].value_counts()
    if len(value_counts) > 10:
        value_counts = value_counts.head(10)
        title_suffix = ' (Top 10)'
        detailed_cols.append(cat)
    else:
        title_suffix = ''
    
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(value_counts)))
    bars = axes1[idx].barh(range(len(value_counts)), value_counts.values, color=colors, edgecolor='black', linewidth=0.5)
    axes1[idx].set_yticks(range(len(value_counts)))
    axes1[idx].set_yticklabels(value_counts.index, fontsize=9)
    axes1[idx].set_xlabel('Count', fontsize=10)
    axes1[idx].set_title(f'{cat}{title_suffix}', fontsize=11, fontweight='bold')
    
    for bar, count in zip(bars, value_counts.values):
        axes1[idx].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, f'{count}', va='center', fontsize=8)
    axes1[idx].grid(axis='x', alpha=0.3, linestyle='--')

for idx in range(n_diags, len(axes1)):
    axes1[idx].axis('off')
plt.suptitle('Categorical Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Detailed categorical feature statistics for high-cardinality categories
for cat in detailed_cols:
    print(f'\n{cat}:')
    print(f'  Unique values: {train[cat].nunique()}')
    print(f'  Most common: {train[cat].value_counts().index[0]} ({train[cat].value_counts().values[0]:,} samples)')
    print(f'  Least common: {train[cat].value_counts().index[-1]} ({train[cat].value_counts().values[-1]:,} samples)')

In [ ]:
## Explore numerical features

n_diags = len(num_columns)
n_cols = min(4, n_diags)
n_rows = (n_diags + n_cols - 1) // n_cols
fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
axes2 = axes2.flatten()

for idx, num_col in enumerate(num_columns):
    data = train[num_col].dropna()
    n_bins = min(30, len(data.unique()))
    counts, bins, patches = axes2[idx].hist(data, bins=n_bins, edgecolor='black', linewidth=0.5, alpha=0.7)
    norm = plt.Normalize(counts.min(), counts.max())
    for patch, count in zip(patches, counts):
        patch.set_facecolor(plt.cm.viridis(norm(count)))
    
    mean_val = data.mean()
    median_val = data.median()
    axes2[idx].axvline(mean_val, color='red', linestyle='-', linewidth=1.5, label=f'Mean: {mean_val:.2f}')
    axes2[idx].axvline(median_val, color='blue', linestyle='--', linewidth=1.5, label=f'Median: {median_val:.2f}')
    
    axes2[idx].set_xlabel('Value', fontsize=9)
    axes2[idx].set_ylabel('Frequency', fontsize=9)
    axes2[idx].set_title(f'{num_col}', fontsize=10, fontweight='bold')
    axes2[idx].legend(fontsize=7, loc='upper right')
    axes2[idx].grid(axis='y', alpha=0.3, linestyle='--')
    
    stats_text = f'Std: {data.std():.2f}\nSkew: {data.skew():.2f}'
    axes2[idx].text(0.95, 0.95, stats_text, transform=axes2[idx].transAxes, fontsize=7, verticalalignment='top',
                   horizontalalignment='right', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

for idx in range(n_diags, len(axes2)):
    axes2[idx].axis('off')
plt.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Numerical feature statistics
print('\nNumerical Features Summary Statistics:')
pd.set_option('display.max_columns', None)
train[num_columns].describe(include='all').round(2)

In [ ]:
## Compare probabilty distribution of features and labels between train, validation and test sets (based on a single fold)

# Create a dataframe with assigned sources (train/validation/test)
df_plot = pd.concat([train[orig_num_columns+orig_cat_columns].assign(Set='Train'), val[orig_num_columns+orig_cat_columns].assign(Set='Validation'),
                     test[orig_num_columns+orig_cat_columns].assign(Set='Test')])
df_plot.insert(3, value=pd.concat([train[target], val[target], pd.Series([None] * len(test), name=target)]), column=target)
df_plot[orig_cat_columns + [target]] = df_plot[orig_cat_columns + [target]].astype('str')
df_plot.reset_index(drop=True, inplace=True)
warnings.filterwarnings('ignore', category=FutureWarning, module='seaborn') # Suppress the specific FutureWarning

# Plot probabilty distribution of features
n_diags = len(orig_num_columns+orig_cat_columns+[target])
n_cols = min(4, n_diags)
n_rows = (n_diags + n_cols - 1) // n_cols
fig3, axes3 = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
fig3.suptitle('Feature distributions in train and validation sets')
axes3 = axes3.flatten()
for i, col in enumerate(orig_num_columns):
    sns.kdeplot(data=df_plot, x=col, ax=axes3[i], hue='Set', common_norm=False, fill=True)
for j, col in enumerate(orig_cat_columns + [target]):
    sns.histplot(data=df_plot, x=col, ax=axes3[i+1+j], hue='Set', bins=len(df_plot[col].unique()),
                 stat='density', discrete=True, multiple='dodge', common_norm=False, shrink=.8)

for idx in range(n_diags, len(axes3)):
    axes3[idx].axis('off')
plt.tight_layout()
plt.show()
del df_plot

# 4. Preprocess data

In [ ]:
## Helping functions for calculating and adding statistical features

def get_stats(df_fold, fold):
    # Calculate global statistic values as replacement value for unknown groups
    globals()[f'global_fold{fold}_stats'] = {'mean': df_fold[target].mean(), 'std': df_fold[target].std(),
                                             'skew': df_fold[target].skew(), 'median': df_fold[target].median(),
                                             'min': df_fold[target].min(), 'max': df_fold[target].max(), 'count': 0}

    # Save statistical features of target values being grouped by feature values
    for stat in ['mean', 'std', 'skew', 'median', 'min', 'max', 'count']:
        globals()[f'fold{fold}_stats_' + stat] = {}
    for column in cat_columns + bin_num_columns:
        for stat in ['mean', 'std', 'skew', 'median', 'min', 'max', 'count']:
            globals()[f'fold{fold}_stats_' + stat][column] = (df_fold.groupby(column)[target].agg(
                [stat])/(len(df_fold) if stat=='count' else 1)).to_dict()[stat]

def target_stats(X, features, st_type, fold):
    fold_global_stats = globals()[f'global_fold{fold}_stats']
    fold_stats = globals()[f'fold{fold}_stats_' + st_type]
    X_stat = pd.DataFrame()
    for c in features:
        X_stat[c] = X[c].map(fold_stats[c]).fillna(fold_global_stats[st_type])
    return X_stat.copy()

In [ ]:
##  Preprocessing pipeline creating function

def create_preprocessor(df_fold, fold):
    # Pipelines for numerical features
    scaler = StandardScaler() # MinMaxScaler() / StandardScaler() / RobustScaler() / 'passthrough'
    prescaler = MinMaxScaler(feature_range=(0.001, 1), clip=True)
    num_imputer = 'passthrough' # SimpleImputer(strategy='median', add_indicator=True) / 'passthrough'
    cat_imputer = SimpleImputer(strategy='constant', fill_value='NA') # SimpleImputer(strategy='most_frequent')
    eps = 1e-6
    scaled_pipeline = Pipeline([('imputer', num_imputer),
                                ('scaling', scaler)])
    log_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                             ('prescaling', prescaler),
                             ('log_trans', FunctionTransformer(func=lambda x: np.log(np.abs(x)+eps), feature_names_out='one-to-one')),
                             ('scaling', scaler)])
    square_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                                ('prescaling', prescaler),
                                ('square_trans', FunctionTransformer(func=np.square, feature_names_out='one-to-one')),
                                ('scaling', scaler)])
    cube_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                              ('prescaling', prescaler),
                              ('cube_trans', FunctionTransformer(func=lambda x: np.power(x, 3), feature_names_out='one-to-one')),
                              ('scaling', scaler)])
    sqrt_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                              ('prescaling', prescaler),
                              ('sqrt_trans', FunctionTransformer(func=lambda x: np.sqrt(np.abs(x)+eps), feature_names_out='one-to-one')),
                              ('scaling', scaler)])
    cbrt_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                              ('prescaling', prescaler),
                              ('cbrt_trans', FunctionTransformer(func=lambda x: np.cbrt(x), feature_names_out='one-to-one')),
                              ('scaling', scaler)])
    kbins_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                               ('kbins', KBinsDiscretizer(n_bins=STAT_N_BINS, strategy='uniform', encode='ordinal', random_state=42)),
                               ('kbins_cast', FunctionTransformer(lambda X: X.astype(np.uint16), feature_names_out='one-to-one'))])
    kmeans_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                                ('scaler', scaler),
                                ('kmeans', KMeans(n_clusters=9, random_state=42)),
                                ('kmeans_cast', FunctionTransformer(lambda X: X.astype(np.uint16), feature_names_out='one-to-one'))])

    nonlinear_transformers = ColumnTransformer([('scaled', scaled_pipeline, num_columns),
                                                ('square', square_pipeline, num_columns),
                                                ('cube', cube_pipeline, num_columns),
                                                ('log', log_pipeline, num_columns),
                                                ('sqrt', sqrt_pipeline, num_columns),
                                                ('cbrt', cbrt_pipeline, num_columns)
                                                ])

    # Pipelines for PCA/LDA on non-linear transformations of numerical features
    pca_pipeline = Pipeline([('nonlin', nonlinear_transformers), ('pca', PCA(n_components=3, random_state=42))])
    lda_pipeline = Pipeline([('nonlin', nonlinear_transformers), ('lda', LinearDiscriminantAnalysis(n_components=2))])

    # Pipelines for categorical features
    ordinal_pipeline = Pipeline([('imputer', cat_imputer),
                                 ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value',
                                    encoded_missing_value=-2, unknown_value=-1, dtype=np.int8))])
    onehot_pipeline = Pipeline([('imputer', cat_imputer),
                                ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
    cat_pipeline = Pipeline([('imputer', cat_imputer)])
    
    # Pipelines for statistical features
    mean_pipeline = Pipeline([('mean', FunctionTransformer(func=lambda x: target_stats(
        x, x.columns, 'mean', fold), feature_names_out='one-to-one'))])
    std_pipeline = Pipeline([('std', FunctionTransformer(func=lambda x: target_stats(
        x, x.columns, 'std', fold), feature_names_out='one-to-one'))])
    skew_pipeline = Pipeline([('skew', FunctionTransformer(func=lambda x: target_stats(
        x, x.columns, 'skew', fold), feature_names_out='one-to-one'))])
    median_pipeline = Pipeline([('median', FunctionTransformer(func=lambda x: target_stats(
        x, x.columns, 'median', fold), feature_names_out='one-to-one'))])
    min_pipeline = Pipeline([('min', FunctionTransformer(func=lambda x: target_stats(
        x, x.columns, 'min', fold), feature_names_out='one-to-one'))])
    max_pipeline = Pipeline([('max', FunctionTransformer(func=lambda x: target_stats(
        x, x.columns, 'max', fold), feature_names_out='one-to-one'))])
    frq_pipeline = Pipeline([('frq', FunctionTransformer(func=lambda x: target_stats(
        x, x.columns, 'count', fold), feature_names_out='one-to-one'))])
    te_pipeline = Pipeline([('te', TargetEncoder(cv=5, shuffle=True, random_state=42))])

    num_stats_pipeline = ColumnTransformer([('mean', mean_pipeline, bin_num_columns),
                                            #('std', std_pipeline, bin_num_columns),
                                            #('skew', skew_pipeline, bin_num_columns),
                                            #('median', median_pipeline, bin_num_columns),
                                            #('min', min_pipeline, bin_num_columns),
                                            #('max', max_pipeline, bin_num_columns),
                                            ('frq', frq_pipeline, bin_num_columns),
                                            #('te', te_pipeline, bin_num_columns),
                                            ])
    cat_stats_pipeline = ColumnTransformer([#('mean', mean_pipeline, cat_columns),
                                            #('std', std_pipeline, cat_columns),
                                            #('skew', skew_pipeline, cat_columns),
                                            #('median', median_pipeline, cat_columns),
                                            #('min', min_pipeline, cat_columns),
                                            #('max', max_pipeline, cat_columns),
                                            ('frq', frq_pipeline, cat_columns),
                                            #('te', te_pipeline, cat_columns),
                                            ])
    
    # Preprocessing pipeline
    preprocessing = ColumnTransformer([## Numerical transformations
                                       ('scaled', scaled_pipeline, num_columns),
                                       #('nonlin', nonlinear_transformers, num_columns),
                                       #('pca', pca_pipeline, num_columns),
                                       #('lda', lda_pipeline, num_columns),
                                       #('cluster', kbins_pipeline, num_columns),
                                       #('kmeans', kmeans_pipeline, num_columns),
    
                                       ## Categorical encoding
                                       ('ordinal', ordinal_pipeline, cat_columns),
                                       #('onehot', onehot_pipeline, cat_columns),
                                       #('categorical', cat_pipeline, cat_columns),
        
                                       ## Statistical transformers
                                       ('num_stats', num_stats_pipeline, bin_num_columns),
                                       ('cat_stats', cat_stats_pipeline, cat_columns),
                                       ]).set_output(transform='pandas')
    return preprocessing

In [ ]:
## Feature selection function based on XGBoost/CatBoost feature importances

def create_feature_selector(max_features):
    # Define base model for feature selection
    xgbr_fs = xgboost.XGBClassifier(device ='cuda' if GPU_ACC else 'cpu', random_state=42, enable_categorical=True, verbosity=0,
                                   n_estimators=10000, learning_rate=0.03, max_depth=8, early_stopping_rounds=300)
    catc_fs = catboost.CatBoostClassifier(auto_class_weights='Balanced',
                                         random_state=42, task_type='GPU' if GPU_ACC else 'CPU', verbose=False)
    model_fs = SelectFromModel(xgbr_fs, max_features=max_features, threshold=1e-6, prefit=False
                              ).set_output(transform='pandas')
    return model_fs

In [ ]:
## Cast categorical features to 'category' for choosen estimators being able to handle it

def cast_categorical(X_train, X_eval, test_prepared, catc, est_id):
    cat_feat = make_column_selector(pattern='ordinal|cluster|onehot|categorical')(X_train)
    X_train[cat_feat] = X_train[cat_feat].astype(str).astype('category' if est_id in EST_IDS_W_CAT_FEAT else 'int16')
    X_eval[cat_feat] = X_eval[cat_feat].astype(str).astype('category' if est_id in EST_IDS_W_CAT_FEAT else 'int16')
    test_prepared[cat_feat] = test_prepared[cat_feat].astype(str).astype('category' if est_id in EST_IDS_W_CAT_FEAT else 'int16')
    if est_id in EST_IDS_W_CAT_FEAT:
        catc.set_params(cat_features=cat_feat if est_id in EST_IDS_W_CAT_FEAT else None)
    return X_train, X_eval, test_prepared, catc

In [ ]:
##  Bucketize numerical features

STAT_N_BINS = 255
bin_num_columns = ['bin_'+col for col in num_columns]
stat_bucket_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
            ('stat_bins', KBinsDiscretizer(n_bins=STAT_N_BINS, strategy='uniform', encode='ordinal', random_state=42))])

def bucketize(X_train, X_eval, test):
    X_train.loc[:,bin_num_columns] = stat_bucket_pipeline.fit_transform(X_train[num_columns]).astype('uint16')
    X_eval.loc[:,bin_num_columns] = stat_bucket_pipeline.transform(X_eval[num_columns]).astype('uint16')
    test.loc[:,bin_num_columns] = stat_bucket_pipeline.transform(test[num_columns]).astype('uint16')
    return X_train, X_eval, test

In [ ]:
## Data preprocessing function allowing a leakage-free data preprocessing of folds

def preprocess(X_train, X_eval, test, y_train, fold, catc, est_id):
    # Bucketize numerical features
    X_train, X_eval, test = bucketize(X_train, X_eval, test)
    
    # Create leakage-free fold statistics
    get_stats(X_train, fold)

    # Create and apply preprocessing pipeline on raw data
    preprocessing = create_preprocessor(X_train, fold)
    X_train = preprocessing.fit_transform(X_train, y_train.astype('str'))
    X_eval = preprocessing.transform(X_eval)
    test_prepared = preprocessing.transform(test)
    print(f'Number of unfiltered features for estimator {EstimatorStr[est_id]} of fold {fold}: {test_prepared.shape[1]}')
    
    # Perform feature selection based on feature importances of choosen model
    MAX_FEAT = None #test_prepared.shape[1] # Max number of features after feature selection
    if MAX_FEAT:
        model_fs = create_feature_selector(MAX_FEAT)
        X_train = model_fs.fit_transform(X_train, y_train, eval_set=[(X_eval, np.array(y_eval))])
        X_eval = model_fs.transform(X_eval)
        test_prepared = model_fs.transform(test_prepared)
    print(f'Number of selected features for estimator {EstimatorStr[est_id]} of fold {fold}: {test_prepared.shape[1]}')

    # Cast categorical features to 'category' for choosen estimators being able to handle it
    X_train, X_eval, test_prepared, catc = cast_categorical(X_train, X_eval, test_prepared, catc, est_id)
    return X_train, X_eval, test_prepared, y_train, catc

In [ ]:
## Merge data sources

ADD_EXTERN_DATA = False       # Extend dataset folds seperately with external dataset

def merge_extern_data(df, external):
    df_columns = [col for col in df.columns if col not in ['id', 'multicat', 'strat_bin']]
    df_merged = pd.concat([df[df_columns], external[df_columns]],
                          ignore_index=True).reset_index(names='id')
    df_merged = df_merged.drop_duplicates()
    return df_merged

# 5. Define model space for ML methods

In [ ]:
## Machine learning models and their hyperparameter search space

GPU_ACC = torch.cuda.is_available()

# Models
svc = SVC(kernel='rbf', class_weight='balanced', random_state=42)
rfc = RandomForestClassifier(class_weight='balanced', random_state=42) #ExtraTreesClassifier
kneigh = KNeighborsClassifier()
gbc = GradientBoostingClassifier(random_state=42)
xgb = xgboost.XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', num_class=3,
                            enable_categorical=True, device='cuda' if GPU_ACC else 'cpu', random_state=0)
ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced', random_state=42), random_state=42)
hgbc = HistGradientBoostingClassifier(scoring='balanced_accuracy', class_weight='balanced', random_state=3)
lgbm = lightgbm.LGBMClassifier(objective='multiclass', metric='multi_logloss', num_class=3, class_weight="balanced",
                               device ='gpu' if GPU_ACC else 'cpu', verbosity=-1, random_state=42)
catc = catboost.CatBoostClassifier(loss_function='MultiClass', eval_metric='MultiClass', auto_class_weights='Balanced',
                                   task_type='GPU' if GPU_ACC else 'CPU', verbose=False, random_state=3)
sgdc = SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42)
rmlp = RealMLP_TD_Classifier(val_metric_name='cross_entropy', device='cuda' if GPU_ACC else 'cpu',
                             random_state=42, verbosity=2)

# Model space
EstimatorStr = {1: 'svc', 2: 'rfc', 3: 'kneigh', 4: 'gbc', 5: 'xgb', 6: 'ada',
                7: 'hgbc', 8: 'lgbm', 9: 'catc', 10: 'sgdc', 11: 'rmlp'}
EstimatorMdl = {1: svc, 2: rfc, 3: kneigh, 4: gbc, 5: xgb, 6: ada, 7: hgbc, 8: lgbm, 9: catc, 10: sgdc, 11: rmlp}

In [ ]:
## Helping function to create parameter grids

def make_param(param_dict, model='est'):
    for elem in param_dict.copy():
        if elem == 'n_components':
            param_dict['pca'+'__'+elem] = param_dict.pop(elem)
        else:
            param_dict[model+'__'+elem] = param_dict.pop(elem)
    return param_dict

In [ ]:
## Tuned hyperparameter sets

# svc parameter
param_single_svc = make_param({}) #
# rfc parameter
param_single_rfc = make_param({#'n_estimators': 1000, 'min_samples_split': 10, 'min_samples_leaf': 2
                               }) # 
# kneight parameter
param_single_kneigh = make_param({}) # 
# gbc parameter
param_single_gbc = make_param({}) # 
# xgb parameter
param_single_xgb = make_param({#'max_depth': 8, 'min_child_weight': 20, 'gamma': 0, 'subsample': 0.9,
                               #'colsample_bytree': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.1,
                               'learning_rate': 0.03, 'n_estimators': 300, 'early_stopping_rounds': 150,
                               }) # 
# ada parameter
param_single_ada = make_param({}) # 
# hgbc parameter
param_single_hgbc = make_param({'max_iter': 10000, 'n_iter_no_change': 10, 'learning_rate': 0.05,
                                'max_leaf_nodes': 23, 'min_samples_leaf': 20, 'l2_regularization': 0.0,
                                'max_depth': None,
                                }) # 
# lgbm parameter
param_single_lgbm = make_param({'num_leaves': 63, 'min_data_in_leaf': 10, 'min_gain_to_split': 0, 'max_depth': 8,
                                'feature_fraction': 0.9, 'bagging_fraction': 0.7, 'lambda_l1': 0, 'lambda_l2': 1,
                                'learning_rate': 0.1, 'n_estimators': 50, 'early_stopping_rounds': 150,
                                }) # 
# catc parameter
param_single_catc = make_param({'n_estimators': 10000, 'learning_rate': 0.015, 'max_depth': 6,
                                'early_stopping_rounds': 150, 'l2_leaf_reg': 9, 'min_data_in_leaf': 1,
                                'bagging_temperature': 1.0, 'random_strength': 1.0, 'max_bin': 1024,
                                }) #
# sgdc parameter
param_single_sgdc = make_param({}) #
# rmlp parameter
param_single_rmlp = make_param({}) #

In [ ]:
## Hyperparameter sets for parameter tuning

# xgb parameter
param_grid_xgb = make_param({# Stage 1: Tree capacity tuning
                             'max_depth': [6,8,10],
                             'min_child_weight': [10,20,30],
                             # Stage 2: Split regularization / Row/column sampling
                             'gamma': [0],
                             'subsample': [0.9],
                             'colsample_bytree': [0.7],
                             # Stage 3: L1/L2 regularization
                             'reg_lambda': [1], #[1, 3, 5, 10],
                             'reg_alpha': [0.1], #[0, 0.01, 0.1, 1],
                             # Stage 4: Learning rate refinement
                             'learning_rate': [0.03],
                             'n_estimators': [300],
                             'early_stopping_rounds': [150],
                             })
# hgbc parameter
param_grid_hgbc = make_param({# Stage 1: Tree capacity
                              'max_leaf_nodes': [23],
                              'learning_rate': [0.05],
                              # Stage 2: Structural regularization
                              'min_samples_leaf': [20],
                              # Stage 3: Leaf value regularization
                              'l2_regularization': [0.0],
                              # Stage 4: Sampling / depth / binning refinement
                              'max_bins': [128, 194, 255],
                              'max_depth': [None],
                              'max_iter': [10000],
                              'n_iter_no_change': [10]
                              })
# lgbm parameter
param_grid_lgbm = make_param({# Stage 1: Tree capacity tuning
                              'num_leaves': [63],
                              'min_data_in_leaf': [10],
                              # Stage 2: Split regularization
                              'min_gain_to_split': [0],
                              'max_depth': [8],
                              # Stage 3: Sampling   
                              'feature_fraction': [0.9],
                              'bagging_fraction': [0.7],
                              # Stage 4: L1/L2 regularization
                              'lambda_l1': [0],
                              'lambda_l2': [1],
                              # Stage 5: Learning rate refinement
                              'learning_rate': [0.03, 0.05, 0.1],
                              'n_estimators': [100, 200, 300],
                              'early_stopping_rounds': [150],
                              })
# catc parameter
param_grid_catc = make_param({# Stage 1: Tree capacity tuning
                              'max_depth': [6],
                              'learning_rate': [0.015],
                              # Stage 2: Structural regularization                       
                              'l2_leaf_reg': [9],
                              'min_data_in_leaf': [1],
                              # Stage 3: Randomness / robustness
                              'bagging_temperature': [1.0, 2.0, 4.0],
                              'random_strength': [0.0, 0.5, 1.0],
                              # Stage 4: Binning / early stopping refinement
                              'max_bin': [1024],
                              'n_estimators': [10000],
                              'early_stopping_rounds': [150],
                              })

# 6. Training

In [ ]:
## Single fitting with tuned parameters or grid search for machine learning methods

TUNING = False        # Choose between single fitting or parameter tuning
EST_IDS = [5,9]     # Choose model(s) {1: 'svc', 2: 'rfc', 3: 'kneigh', 4: 'gbc', 5: 'xgb', 6: 'ada',
                      #                  7: 'hgbc', 8: 'lgbm', 9: 'catc', 10: 'sgdc', 11: 'rmlp'}
EST_IDS_W_EARLYSTOPPING = [5,7,8,9,11]
EST_IDS_W_CAT_FEAT = [5,7,8,9,11]
MULTI_SEED = False
val_preds = pd.DataFrame(trainval[target])
val_probs = np.full((len(EST_IDS), len(trainval), 3), np.nan, dtype=np.float32)
test_preds = pd.DataFrame()
test_probs = []

for idx, est_id in enumerate(EST_IDS):
    start_time = time.time()
    bal_acc_train_ls = []
    bal_acc_val_ls = []
    test_probs_est = []
    
    # Define pipeline
    pipeline = Pipeline([('est', EstimatorMdl[est_id])])

    # Cross-validation configuration w/wo extended stratification
    cv_gen = skf.split(trainval, trainval[strat_col], trainval[group_col])

    # Fitting or tuning on train dataset with k-fold cross-validation
    param = globals()[f'param_grid_{EstimatorStr[est_id]}' if TUNING else f'param_single_{EstimatorStr[est_id]}']
    if TUNING:
        eval_set = {}
        if est_id in EST_IDS_W_EARLYSTOPPING:
            # Split data into train and validation set and set configuration parameters if early stopping configured
            train_index, eval_index = next(skf.split(trainval, trainval[strat_col], trainval[group_col]))
            X_train, X_eval = trainval.iloc[train_index], trainval.iloc[eval_index]
            
            # Add extern data to fold train dataset
            if ADD_EXTERN_DATA:
                X_train = merge_extern_data(X_train, extern_data)
            y_train, y_eval = X_train[target], X_eval[target]
            strat_train = X_train[strat_col]
            group_train = X_train[group_col]

            # Preprocess fold data
            X_train, X_eval, test_prepared, y_train, catc = preprocess(
                X_train, X_eval, test, y_train, 'train', catc, est_id)

            # Set model specific early stopping parameters
            if (est_id==7) | (est_id==11):
                eval_set['est__X_val'], eval_set['est__y_val'] = X_eval, np.array(y_eval)
            else:
                eval_set['est__eval_set'] = [(X_eval, np.array(y_eval))]
            if est_id==5:
                eval_set['est__verbose'] = 0
            
            cv_gen = skf.split(X_train, strat_train, group_train)
        else:
            # Preprocess fold data
            X_train, _, test_prepared, y_train, catc = preprocess(
                trainval, trainval, test, trainval[target], 'train', catc, est_id)
        if est_id==5:
            eval_set['est__sample_weight'] = compute_sample_weight(class_weight="balanced", y=y_train)
            
        # Tune model
        grid = GridSearchCV(pipeline, param, scoring='balanced_accuracy', verbose=3, cv=cv_gen)
        grid.fit(X_train, np.array(y_train), **eval_set)
        print(grid.best_params_)
        print(grid.cv_results_)
        pipeline_tune = grid.best_estimator_

        # Store predictions and model
        val_preds.loc[eval_index, f'pred_{EstimatorStr[est_id]}'] = pipeline_tune.predict(X_eval)
        val_preds = val_preds.dropna()
        val_probs[idx, eval_index, :] = pipeline_tune.predict_proba(X_eval)
        val_probs = val_probs[:, ~np.isnan(val_probs[0]).any(axis=-1), :]
        test_preds[f'pred_{EstimatorStr[est_id]}'] = pipeline_tune.predict(test_prepared).ravel()
        test_probs_est.append(pipeline_tune.predict_proba(test_prepared))
        globals()[f'model1_{EstimatorStr[est_id]}'] = pipeline_tune
    else:
        # Training with k-fold cross-validation
        for i, (train_index, eval_index) in enumerate(cv_gen):
            fold = str(i+1)
            
            # Split data into train and validation set
            X_train, X_eval = trainval.iloc[train_index], trainval.iloc[eval_index]
            
            # Add extern data to fold train dataset
            if ADD_EXTERN_DATA:
                X_train = merge_extern_data(X_train, extern_data)

            # Take labels
            y_train, y_eval = X_train[target], X_eval[target]

            # Preprocess fold data
            X_train, X_eval, test_prepared, y_train, catc = preprocess(
                X_train, X_eval, test, y_train, fold, catc, est_id)

            # Clone selected pipeline, set configuration parameters and train model
            pipeline_train = deepcopy(pipeline)
            pipeline_train.set_params(**param)
            if MULTI_SEED:
                pipeline_train.set_params(est__random_state=(42+1*i))  
            eval_set = {}
            if est_id in EST_IDS_W_EARLYSTOPPING:
                if (est_id==7) | (est_id==11):
                    eval_set['est__X_val'], eval_set['est__y_val'] = X_eval, np.array(y_eval)
                else:
                    eval_set['est__eval_set'] = [(X_eval, np.array(y_eval))]
                if est_id==5:
                    eval_set['est__verbose'] = 0
                elif est_id==8:
                    eval_set['est__callbacks']=[lightgbm.early_stopping(param_single_lgbm['est__early_stopping_rounds'])]
            if est_id==5:
                eval_set['est__sample_weight'] = compute_sample_weight(class_weight="balanced", y=y_train)
            pipeline_train.fit(X_train, np.array(y_train), **eval_set)

            # Calculate and show balanced accuracy scores after training of each fold
            train_score = balanced_accuracy_score(np.array(y_train), pipeline_train.predict(X_train))
            eval_preds = pipeline_train.predict(X_eval)
            val_score = balanced_accuracy_score(np.array(y_eval), eval_preds)
            
            # Store oof predictions and the scores for each fold
            val_preds.loc[eval_index, f'pred_{EstimatorStr[est_id]}'] = eval_preds
            val_probs[idx, eval_index, :] = pipeline_train.predict_proba(X_eval)
            bal_acc_train_ls.append(train_score)
            bal_acc_val_ls.append(val_score)

            # Store predictions for test set
            test_preds[f'pred{fold}_{EstimatorStr[est_id]}'] = pipeline_train.predict(test_prepared).ravel()
            test_probs_est.append(pipeline_train.predict_proba(test_prepared))
                                                           
            # Save trained pipeline
            globals()[f'model{fold}_{EstimatorStr[est_id]}'] = pipeline_train
            
            print(f'Estimator: {EstimatorStr[est_id]} of fold {fold} is fitted')
            print(f'Train balanced accuracy score: {train_score}')
            print(f'Val balanced accuracy score: {val_score}')
            print(f'Elapsed time: {int(time.time() - start_time)} [s]')
            print('-'*10)

        print(f'Average train balanced accuracy score of {EstimatorStr[est_id]} estimator over all folds: {sum(bal_acc_train_ls)/len(bal_acc_train_ls)}')
        print(f'Average val balanced accuracy score of {EstimatorStr[est_id]} estimator over all folds: {sum(bal_acc_val_ls)/len(bal_acc_val_ls)}')
        print('-'*80)
        print('-'*80)
    test_probs.append(np.stack(test_probs_est).mean(axis=0))

# Stack the list of test probabilities per estimator
test_probs = np.stack(test_probs)

In [ ]:
## Stack or find weights for each estimator

STACKING = False
WEIGHTING = True

def find_best_balanced_accuracy_weights(val_probs, test_probs, y_true, n_trials=1000):
    n_estimators, n_samples, n_classes = val_probs.shape
    best_score = -1.0
    best_weights = None
    val_blend = None

    # Include uniform weights
    candidate_weights = [np.ones(n_estimators) / n_estimators]

    if WEIGHTING:
        # Include single-estimator solutions
        for i in range(n_estimators):
            w = np.zeros(n_estimators)
            w[i] = 1.0
            candidate_weights.append(w)
    
        # Include random weights on simplex
        rng = np.random.default_rng(seed=42)
        random_weights = rng.dirichlet(np.ones(n_estimators), size=n_trials)
        candidate_weights.extend(random_weights)

    # Test each weight distribution and select best one
    for weights in candidate_weights:
        blend = np.tensordot(weights, val_probs, axes=(0, 0))
        preds = blend.argmax(axis=1)
        score = balanced_accuracy_score(y_true, preds)

        if score > best_score:
            best_score = score
            best_weights = weights.copy()
            val_blend = blend.copy()

    # Calculate test blend with nest weight distibution
    test_blend = np.tensordot(weights, test_probs, axes=(0, 0))
    return best_weights, best_score, val_blend, test_blend

def stack_models(val_probs, test_probs, y_true):
    meta_train = val_probs.transpose(1, 0, 2).reshape(val_probs.shape[1], -1)
    meta_test = test_probs.transpose(1, 0, 2).reshape(test_probs.shape[1], -1)
    #meta_model = LogisticRegression(max_iter=2000, class_weight='balanced', C=100, l1_ratio=0.0, random_state=42)
    meta_model = HistGradientBoostingClassifier(scoring='balanced_accuracy', class_weight='balanced',
                                                learning_rate=0.05,
                                                #max_leaf_nodes=15, min_samples_leaf=200,
                                                #l2_regularization=1.0,
                                                max_iter=10000, n_iter_no_change=40,
                                                random_state=42)
    meta_model.fit(meta_train, y_true)
    stack_val_probs = meta_model.predict_proba(meta_train)
    stack_test_probs = meta_model.predict_proba(meta_test)
    return stack_val_probs, stack_test_probs

if STACKING and len(EST_IDS)>1:
    val_blend, test_blend = stack_models(
        val_probs, test_probs, np.array(val_preds[target]))
else:
    best_weights, best_score, val_blend, test_blend = find_best_balanced_accuracy_weights(
        val_probs, test_probs, np.array(val_preds[target]))
    print(f'Estimator weights: {best_weights}')

# 7. Evaluation

In [ ]:
## Calculate balanced accuracy scores for each estimators and mean balanced accuracy score over all estimators based on accumulated Out-of-Fold (oof) samples

# Balanced accuracy scores for each estimator based on accumulated oof samples
for est_id in EST_IDS:
    est_oof_score = balanced_accuracy_score(val_preds[target], val_preds[f'pred_{EstimatorStr[est_id]}'])
    print(f'Balanced accuracy score over all oof samples for {EstimatorStr[est_id]} estimator: {est_oof_score}')
    print('-'*120)

# Balanced accuracy score on mean value of ensemble predictions based on accumulated oof samples
val_preds['pred_score'] = val_preds.filter(like='pred').mean(axis=1).round().astype(np.uint8)
val_preds['prob_score'] = val_blend.argmax(axis=-1)

overall_oof_score = balanced_accuracy_score(val_preds[target], val_preds['pred_score'])
overall_oof_score_prob = balanced_accuracy_score(val_preds[target], val_preds['prob_score'])

print(f'Overall balanced accuracy score over all oof samples and all estimators: {overall_oof_score}')
print(f'Overall balanced accuracy score over all oof samples and all estimators based on probabilitites: {overall_oof_score_prob}')
print('-'*120)
print('Show predictions and labels of validation dataset: ')
print(val_preds)

In [ ]:
## Show balanced accuracy scores of subcategory subsets

subcats = []
subcat_bal_acc_scores  = []
for cat in orig_cat_columns:
    for subcat in np.sort([cat for cat in trainval[cat].unique() if cat not in [np.nan]]):
        if trainval[target][trainval[cat] == subcat].nunique() == 3:
            subcats.append(cat+'_'+str(subcat))
            val_filtered = val_preds[target][trainval[cat] == subcat]
            val_labels_filtered = val_preds['pred_score'][trainval[cat] == subcat]
            subcat_bal_acc_scores.append(balanced_accuracy_score(val_filtered, val_labels_filtered))

# Create bar chart
fig1, ax1 = plt.subplots(figsize=(10, 8))
cmap = plt.get_cmap('viridis')
rescale = lambda x: (x - np.min(x)) / (np.max(x) - np.min(x))
normalized_values = rescale(subcat_bal_acc_scores)
ax1.barh(subcats, np.array(subcat_bal_acc_scores)-overall_oof_score, color=cmap(normalized_values), left=overall_oof_score)
plt.title(f'Balanced accuracy scores of subcategory subsets compared to the average balanced accuracy score')
plt.xlabel('Balanced accuracy score')
plt.ylabel('Subcategories')
ax1.axvline(x=overall_oof_score, color='green', linestyle='-.')
plt.tight_layout()
ax1.tick_params(left=False, bottom=True)

In [ ]:
## Show worse balanced accuracy scores of multicategory subsets

if not TUNING:
    multicats = []
    multicat_bal_acc_scores = []
    for multicat in trainval['multicat'].unique():
        if trainval[target][trainval['multicat'] == multicat].nunique() == 3:
            multicats.append(multicat)
            val_filtered = val_preds[target][trainval['multicat'] == multicat]
            val_labels_filtered = val_preds['pred_score'][trainval['multicat'] == multicat]
            multicat_bal_acc_scores.append(balanced_accuracy_score(val_filtered, val_labels_filtered))
    multicats = multicat_encoder.inverse_transform(multicats)
    sorted_multicat_bal_acc_scores, sorted_multicats = zip(*sorted(zip(multicat_bal_acc_scores, multicats), reverse=False))
    
    # Create bar chart
    fig2, ax2 = plt.subplots(figsize=(10, 8))
    n_top = 10
    cmap = plt.get_cmap('viridis')
    rescale = lambda x: (x - np.min(x)) / (np.max(x) - np.min(x))
    normalized_values = rescale(sorted_multicat_bal_acc_scores[:n_top])
    ax2.barh(sorted_multicats[:n_top], np.array(sorted_multicat_bal_acc_scores[:n_top])-overall_oof_score,
             color=cmap(normalized_values), left=overall_oof_score)
    plt.title(f'Worse {n_top} balanced accuracy scores of multicategory subsets')
    plt.xlabel('Balanced accuracy score')
    plt.ylabel('Subcategories')
    plt.tight_layout()
    ax2.tick_params(left=False, bottom=True)

In [ ]:
## Show feature importances

if est_id in [5, 8, 9] and not TUNING:
    # Sort feature names and their importances
    categories = X_train.columns
    values = globals()[f'model{FOLDS}_{EstimatorStr[est_id]}'][-1].feature_importances_+1e-4
    sorted_values, sorted_categories = zip(*sorted(zip(values,categories), reverse=False))
    
    # Plot feature importances
    fig3, ax3 = plt.subplots(figsize=(10, 20))
    normalized_values = rescale(np.log(sorted_values))
    ax3.barh(sorted_categories, sorted_values, color=cmap(normalized_values), log=True)
    plt.title(f'Feature Importances (Magnitude)')
    plt.xlabel('Logarithmic importance score')
    plt.ylabel('Features')
    plt.tight_layout()
    ax3.tick_params(left=False, bottom=True)

# 8. Submission

In [ ]:
## Test prediction & submission 

PROB_MEAN = True
submission_df = test[['id']].copy()

# Take the mean value of prediction over all estimators and folds
if PROB_MEAN:
    submission_df[target] = pd.DataFrame(test_blend.argmax(axis=-1)).replace(reverse_label_mapping)
else:
    submission_df[target] = test_preds.mean(axis=1).round().astype(np.uint8).replace(reverse_label_mapping)

# Write dataframe to .csv file
submission_df.to_csv('submission.csv', index=False)
print('✅ submission.csv saved!')
submission_df